# DoorKey Stronger Experiment

This notebook fixes the main weakness of the Empty-room experiment: the task was too easy and too deterministic. DoorKey requires navigation, picking up a key, opening a door, and reaching the goal.

In [ ]:
!git clone https://github.com/mehddii/skip-vae-rl.git
%cd skip-vae-rl
!git pull
!pip install -q uv
!uv sync --extra notebook

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Representation Training

Train a normal VAE and a regularized Skip-VAE on DoorKey observations. The regularized Skip-VAE uses skip dropout and skip scaling so the decoder cannot rely entirely on copied encoder features.

In [ ]:
!uv run python -m skip_vae_rl.train_vae --config configs/vae_doorkey.yaml
!uv run python -m skip_vae_rl.train_vae --config configs/skip_vae_regularized_doorkey.yaml

## RL Matrix

This is the important comparison: raw pixels, frozen encoders, and fine-tuned encoders on a harder task.

In [ ]:
!uv run python -m skip_vae_rl.train_rl --config configs/ppo_doorkey_raw.yaml
!uv run python -m skip_vae_rl.train_rl --config configs/ppo_doorkey_vae_frozen.yaml
!uv run python -m skip_vae_rl.train_rl --config configs/ppo_doorkey_vae_finetune.yaml
!uv run python -m skip_vae_rl.train_rl --config configs/ppo_doorkey_skip_regularized_frozen.yaml
!uv run python -m skip_vae_rl.train_rl --config configs/ppo_doorkey_skip_regularized_finetune.yaml

## Figures And Table

In [ ]:
!uv run python -m skip_vae_rl.visualize_latent --config configs/latent_doorkey_vae.yaml
!uv run python -m skip_vae_rl.visualize_latent --config configs/latent_doorkey_skip_regularized.yaml
!uv run python -m skip_vae_rl.aggregate_results

In [ ]:
import pandas as pd
pd.read_csv("reports/results_summary.csv")

In [ ]:
from pathlib import Path
from IPython.display import Image, display

for path in sorted(Path("reports/figures").glob("*doorkey*.png")):
    print(path)
    display(Image(filename=str(path)))

In [ ]:
!zip -r doorkey_skip_vae_results.zip runs reports